# 02b — Column generation, walked through (live)

Same content as `02a`, but now **you supply the graph** and a real
Dirac-3 call drives the pricing subproblem. The notebook also writes
every Dirac response to `notebooks/runs/<timestamp>_cg/` in the same
schema as `RF-branching/instances/...`, so you can replay this exact
run offline later.

> **Scaling caveat.** ER(20, 0.5) on cloud Dirac is ~30 s/call; ER(50, 0.5)
> is ~5 min/call. Start small.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import _demo_utils as U
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from quantum_colgen.column_generation import column_generation
from quantum_colgen.pricing.classical_lp import ClassicalLPPricingOracle
from quantum_colgen.graphs import erdos_renyi


## 1. Configuration

In [ ]:
# ── Backend toggle ──────────────────────────────────────────────────
BACKEND = "replay"   # "replay" | "cloud" | "direct"

# Direct hardware endpoint (only used when BACKEND == "direct").
# Skill-documented default 172.18.41.79 is offline; 172.18.41.228 is current.
DIRECT_IP = "172.18.41.228"
DIRECT_PORT = 50051

# Save every live Dirac call to notebooks/runs/<ts>_cg/raw_samples/?
SAVE_RUN = True

# Graph specification (ignored in replay mode — we use the bundled ER(20, 0.7))
NODES, EDGE_PROB, SEED = 20, 0.5, 42
# ────────────────────────────────────────────────────────────────────

U.load_dotenv_if_present()


In [ ]:
# Build the graph
if BACKEND == "replay":
    psp = U.load_psp(1)
    G = U.psp_to_graph(psp)
    layout = U.psp_to_layout(psp)
    print(f"Replay graph: {psp['instance_id']}  n={psp['n']}  m={psp['m']}")
else:
    G = erdos_renyi(NODES, EDGE_PROB, seed=SEED)
    layout = nx.kamada_kawai_layout(G)
    layout = {int(v): (float(p[0]), float(p[1])) for v, p in layout.items()}
    print(f"User graph: ER({NODES}, {EDGE_PROB}, seed={SEED})  "
          f"n={G.number_of_nodes()}  m={G.number_of_edges()}")


In [ ]:
# Set up the run directory (live modes only)
run_dir = None
if BACKEND in ("cloud", "direct") and SAVE_RUN:
    run_dir = U.new_run_dir(label=f"cg_{BACKEND}")
    U.save_run_metadata(run_dir, G, {
        "backend": BACKEND, "n": G.number_of_nodes(), "m": G.number_of_edges(),
        "seed": SEED, "edge_prob": EDGE_PROB,
    })
    print(f"Saving raw Dirac responses to {run_dir}")


## 2. Wire the oracle and run CG

In [ ]:
# Inner Dirac oracle (replay / cloud / direct)
inner = U.make_dirac_oracle(
    BACKEND, method="gibbons",
    num_samples=100, multi_prune=True, randomized_rounding=True,
    num_random_rounds=10, random_seed=42,
    direct_ip_address=DIRECT_IP, direct_port=DIRECT_PORT,
)

# Wrap with on-disk capture and in-memory history
oracle = inner
if run_dir is not None:
    oracle = U.CapturingPricingOracle(oracle, run_dir=run_dir)
oracle = U.HistoryPricingOracle(oracle)

print(f"Oracle stack: HistoryPricingOracle → "
      f"{'CapturingPricingOracle → ' if run_dir else ''}"
      f"{type(inner).__mro__[1].__name__}({getattr(inner, 'backend', '?')})")


In [ ]:
import time
t0 = time.monotonic()
chi, coloring, stats = column_generation(G, oracle, max_iterations=50, verbose=True)
wall = time.monotonic() - t0

print(f"\nχ = {chi}   iterations = {stats.get('iterations')}   wall = {wall:.1f}s")
timer = inner.timer
print(f"Oracle: {timer.summary()}")


## 3. Iteration browser

Step through each CG iteration and inspect duals + extracted columns.

In [ ]:
# Use ipywidgets if available, otherwise print a static table
try:
    from ipywidgets import IntSlider, interact

    def show_iter(i: int = 0):
        h = oracle.history[i]
        sub_g = nx.Graph()
        sub_g.add_nodes_from(h["node_list"])
        sub_g.add_edges_from(h["graph_edges"])
        # use the layout from the full graph for consistency
        sub_layout = {v: layout[v] for v in sub_g.nodes() if v in layout}
        if not sub_layout:
            sub_layout = nx.kamada_kawai_layout(sub_g)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
        U.draw_graph_duals(sub_g, h["dual_vars"], sub_layout, ax=axes[0],
                           title=f"Iter {i}: duals")
        # plot the union of columns as one figure
        if h["columns"]:
            U.draw_graph_duals(sub_g, h["dual_vars"], sub_layout, ax=axes[1],
                               title=f"Iter {i}: {len(h['columns'])} columns extracted",
                               show_values=False)
            for k, c in enumerate(h["columns"][:6]):
                color = U.IS_PALETTE[k % len(U.IS_PALETTE)]
                xs = [sub_layout[v][0] for v in c if v in sub_layout]
                ys = [sub_layout[v][1] for v in c if v in sub_layout]
                axes[1].scatter(xs, ys, s=420, facecolors="none",
                                edgecolors=color, linewidths=2.5)
        plt.show()
        print(f"  columns: {[sorted(c) for c in h['columns']]}")

    interact(show_iter, i=IntSlider(min=0, max=max(0, len(oracle.history)-1),
                                    step=1, value=0))
except ImportError:
    print("ipywidgets not installed — falling back to static iteration table.")
    for i, h in enumerate(oracle.history):
        print(f"iter {i}: {len(h['columns'])} cols, "
              f"sizes={[len(c) for c in h['columns']]}, "
              f"dual_sum={h['dual_vars'].sum():.2f}")


## 4. Classical-CG side-by-side

In [ ]:
classical_oracle = ClassicalLPPricingOracle()
t0 = time.monotonic()
chi_c, coloring_c, stats_c = column_generation(
    G, classical_oracle, max_iterations=200, verbose=False)
wall_c = time.monotonic() - t0

cmp = pd.DataFrame([
    {"oracle": "Quantum (Dirac)", "χ": chi, "iterations": stats.get("iterations"),
     "wall_s": f"{wall:.1f}",
     "cols/call": f"{inner.timer.summary().get('avg_columns_per_call', 0):.1f}",
     "api_calls": inner.timer.summary().get('num_api_calls')},
    {"oracle": "Classical (LP)", "χ": chi_c, "iterations": stats_c.get("iterations"),
     "wall_s": f"{wall_c:.1f}",
     "cols/call": f"{classical_oracle.timer.summary().get('avg_columns_per_call', 0):.1f}"
                  if hasattr(classical_oracle, 'timer') else "—",
     "api_calls": classical_oracle.timer.summary().get('num_api_calls')
                  if hasattr(classical_oracle, 'timer') else "—"},
])
cmp


## 5. Final coloring

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
U.draw_coloring(G, coloring, layout, ax=axes[0],
                title=f"Quantum CG — χ={chi}")
U.draw_coloring(G, coloring_c, layout, ax=axes[1],
                title=f"Classical CG — χ={chi_c}")
plt.show()


## 6. Replay-your-own-run round-trip

If `SAVE_RUN=True` was set above and we ran in cloud or direct mode,
the cell below rebuilds a `ReplayDiracOracle` from the saved
pickles and re-runs CG entirely offline. The χ should match (or be very
close — non-strict replay tolerates small dual-vector drift).

In [ ]:
if run_dir is not None and run_dir.exists() and \
        (run_dir / "raw_samples" / "index.jsonl").exists():
    print(f"Replaying from {run_dir}")
    replay_oracle = U.replay_oracle_from_run(run_dir, method="gibbons")
    chi_r, coloring_r, stats_r = column_generation(
        G, replay_oracle, max_iterations=stats.get("iterations", 50) + 5,
        verbose=False,
    )
    print(f"Live   χ = {chi}   iters = {stats.get('iterations')}   "
          f"wall = {wall:.1f}s")
    print(f"Replay χ = {chi_r}   iters = {stats_r.get('iterations')}   "
          f"wall = {stats_r.get('time_total', 0):.3f}s")
    print(f"⇒ replay reproduced this run offline.")
else:
    print("No saved run available (BACKEND=replay or SAVE_RUN=False).")
    print("Re-run with BACKEND='cloud' or 'direct' and SAVE_RUN=True to capture.")
